# Attach gp3 and io2 volumes to the same instance, run fio, and confirm the IOPS you paid for

A database lead has read three blog posts that say io2 is faster than gp3 and is pushing for an io2 migration that would 2x the storage bill. You are the engineer who has to settle the debate with a benchmark, not opinion. You have one session to stand up a single instance with both volume types attached, run identical fio workloads against each, and surface the sustained-IOPS numbers so the team can pick the right tradeoff based on published volume specs the instance actually delivers, not the marketing pages.

You will build one t4g.small EC2 instance in the default VPC with two extra EBS volumes:

- **Architecture A.** 8 GiB gp3 volume at the gp3 default of 3000 IOPS and 125 MB/s throughput, attached at `/dev/sdf` (Nitro remaps to `/dev/nvme1n1`).
- **Architecture B.** 8 GiB io2 volume provisioned at 1000 IOPS, attached at `/dev/sdg` (Nitro remaps to `/dev/nvme2n1`).

Then you drive `fio` against each volume via SSM Session Manager with the same 4 KiB random-read workload and 60-second runtime. gp3 should land near 3000 IOPS (its baseline); io2 should land near 1000 IOPS (exactly what you provisioned). The point of the lab is that io2 delivers exactly what you provisioned and not more, while gp3 delivers its 3000 IOPS baseline for free.

**Time.** Plan for about 60 minutes hands-on. Instance launch and status checks take 2-3 minutes; each fio run takes ~75 seconds; the rest is reading specs and reading numbers.

**Cost.** This lab costs about three cents if you wrap it up in an hour. The instance is the dominant cost line at 1.7 cents per hour; the two volumes together are about a third of a cent. The cost trap is the volumes outliving the instance: if you terminate without detaching them first, they sit in `available` state and bill per GB-month indefinitely. Cleanup handles the order for you, but the lab is explicit about it because this is a real production gotcha.

This lab maps to AWS SAA-C03 Domain 3 (Design High-Performing Architectures, 24% of exam weight).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md.

!pip install --quiet labrun-checks==0.6.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12. Architecture suffixes -gp3 and -io2 disambiguate the two
# competing volumes per blueprint section 21 (Compare-it).

import atexit
import datetime as dt
import getpass
import json
import time

import boto3
from botocore.exceptions import ClientError
from IPython.display import clear_output

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "ebs-volume-types-and-iops"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

EC2_ROLE_NAME = f"labrun-{LAB_ID}-ec2-role"
INSTANCE_PROFILE_NAME = f"labrun-{LAB_ID}-instance-profile"
SG_NAME = f"labrun-{LAB_ID}-sg"
EC2_NAME = f"labrun-{LAB_ID}-ec2"
GP3_NAME = f"labrun-{LAB_ID}-gp3"
IO2_NAME = f"labrun-{LAB_ID}-io2"

# Device names. EBS attach uses logical names (sdf, sdg); the Linux kernel
# on Nitro remaps them to /dev/nvme1n1, /dev/nvme2n1 in attach order.
GP3_DEVICE = "/dev/sdf"
IO2_DEVICE = "/dev/sdg"
GP3_NITRO_DEVICE = "/dev/nvme1n1"
IO2_NITRO_DEVICE = "/dev/nvme2n1"

# Runtime state captured as the lab proceeds. Module-level so checkpoint
# and observe cells can read them.
LAB_STATE = {
    "instance_id": None,
    "instance_az": None,
    "gp3_volume_id": None,
    "io2_volume_id": None,
    "sg_id": None,
    "gp3_fio_iops": None,
    "io2_fio_iops": None,
}

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm region.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"SAA labs run in {REGION} (N. Virginia).")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# The EC2 instance is critical (hourly billed); it tears down first. The
# gp3 volume is set DeleteOnTermination=True at instance launch so it
# vanishes when the instance terminates. The io2 volume is attached
# separately after launch, so the cleanup cell detaches and deletes it
# manually before run_cleanup terminates the instance.

CLEANUP_MANIFEST: list[CleanupResource] = []


def _rebuild_manifest():
    """Rebuild the manifest in critical-first reverse-creation order from
    the IDs captured in LAB_STATE. Called by every task cell after a
    resource is created so atexit always sees the latest snapshot."""
    CLEANUP_MANIFEST.clear()
    if LAB_STATE["instance_id"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="ec2_instance",
                id=LAB_STATE["instance_id"],
                region=REGION,
                cli_delete_command=(
                    f"aws ec2 terminate-instances --instance-ids {LAB_STATE['instance_id']}"
                ),
            )
        )
    if LAB_STATE["sg_id"]:
        CLEANUP_MANIFEST.append(
            CleanupResource(
                type="ec2_security_group",
                id=LAB_STATE["sg_id"],
                region=REGION,
                cli_delete_command=f"aws ec2 delete-security-group --group-id {LAB_STATE['sg_id']}",
            )
        )
    CLEANUP_MANIFEST.append(
        CleanupResource(
            type="iam_instance_profile",
            id=INSTANCE_PROFILE_NAME,
            parent=EC2_ROLE_NAME,
            region=REGION,
            cli_delete_command=(
                f"aws iam remove-role-from-instance-profile "
                f"--instance-profile-name {INSTANCE_PROFILE_NAME} "
                f"--role-name {EC2_ROLE_NAME} && "
                f"aws iam delete-instance-profile "
                f"--instance-profile-name {INSTANCE_PROFILE_NAME}"
            ),
        )
    )
    CLEANUP_MANIFEST.append(
        CleanupResource(
            type="iam_role",
            id=EC2_ROLE_NAME,
            region=REGION,
            cli_delete_command=f"aws iam delete-role --role-name {EC2_ROLE_NAME}",
        )
    )


_rebuild_manifest()


def _detach_and_delete_io2():
    """Detach + delete the io2 volume manually. labrun-checks 0.6.0 has no
    ebs_volume adapter; the cleanup cell handles it inline."""
    if not LAB_STATE.get("io2_volume_id"):
        return
    ec2c = boto3.client(
        "ec2",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    try:
        ec2c.detach_volume(VolumeId=LAB_STATE["io2_volume_id"], Force=True)
    except ClientError as e:
        code = e.response.get("Error", {}).get("Code", "")
        if code not in ("InvalidVolume.NotFound", "IncorrectState"):
            pass
    # Wait up to 90 seconds for the volume to be available, then delete.
    deadline = time.time() + 90
    while time.time() < deadline:
        try:
            v = ec2c.describe_volumes(VolumeIds=[LAB_STATE["io2_volume_id"]])["Volumes"][0]
            if v.get("State") in ("available", "error"):
                break
        except ClientError as e:
            if e.response.get("Error", {}).get("Code") == "InvalidVolume.NotFound":
                return
            break
        time.sleep(5)
    try:
        ec2c.delete_volume(VolumeId=LAB_STATE["io2_volume_id"])
    except ClientError as e:
        if e.response.get("Error", {}).get("Code") != "InvalidVolume.NotFound":
            pass


def _atexit_cleanup():
    try:
        _detach_and_delete_io2()
        _rebuild_manifest()
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first, or")
    print("remove them with the AWS CLI commands the cleanup cell prints,")
    print("then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to provision the instance and volumes.")

## Task 1: Launch the instance with the gp3 volume inline, then attach the io2 volume

Both volumes have to live in the same AZ as the instance because EBS volumes are AZ-scoped: `attach_volume` fails with `InvalidParameterValue` across AZs. The lab pins the instance to the AZ returned by the default VPC's first subnet, then creates the io2 in that same AZ.

Three pieces, in order:

1. **IAM**: create the role with the EC2 trust policy, attach the AWS-managed `AmazonSSMManagedInstanceCore` policy so SSM Session Manager can drive in-instance commands. Create the instance profile and add the role to it.
2. **Security group**: no inbound rules (all access is via SSM, never SSH). Default outbound rule is kept (the SSM agent calls the SSM endpoints).
3. **Launch**: pick the first default-VPC subnet, capture its AZ, create the io2 volume in that AZ, then call `run_instances` with the gp3 inline via `BlockDeviceMappings` (DeleteOnTermination=True so it vanishes with the instance). After the instance reaches `running`, call `attach_volume` to attach the io2 at `/dev/sdg`.

Both volumes are tagged with the lab slug at creation time so the orphan scan and tag audit pick them up. The gp3 inherits the volume tag from `run_instances` TagSpecifications with ResourceType=`volume`.

The observe cell after the build polls until the instance is `Status=ok` (post-user-data, SSM agent up) and both volumes show as `in-use`.

In [ ]:
# NBVAL_SKIP
# Task 1: IAM, security group, default VPC subnet lookup, io2 volume,
# instance launch with gp3 inline + io2 attach.

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
ec2 = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
ssm = boto3.client(
    "ssm",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# IAM role with EC2 assume-role trust + AmazonSSMManagedInstanceCore.
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "ec2.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}
try:
    iam.create_role(
        RoleName=EC2_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") != "EntityAlreadyExists":
        raise
iam.attach_role_policy(
    RoleName=EC2_ROLE_NAME,
    PolicyArn="arn:aws:iam::aws:policy/AmazonSSMManagedInstanceCore",
)
try:
    iam.create_instance_profile(
        InstanceProfileName=INSTANCE_PROFILE_NAME,
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") != "EntityAlreadyExists":
        raise
try:
    iam.add_role_to_instance_profile(
        InstanceProfileName=INSTANCE_PROFILE_NAME, RoleName=EC2_ROLE_NAME
    )
except ClientError as e:
    if e.response.get("Error", {}).get("Code") != "LimitExceeded":
        raise

# Default VPC first subnet -> AZ for both volumes and the instance.
default_vpc = ec2.describe_vpcs(
    Filters=[{"Name": "is-default", "Values": ["true"]}],
)["Vpcs"][0]
default_vpc_id = default_vpc["VpcId"]
subnets = ec2.describe_subnets(
    Filters=[{"Name": "vpc-id", "Values": [default_vpc_id]}],
)["Subnets"]
subnet = subnets[0]
subnet_id = subnet["SubnetId"]
LAB_STATE["instance_az"] = subnet["AvailabilityZone"]
print(f"Default VPC: {default_vpc_id}")
print(f"Subnet:      {subnet_id} ({LAB_STATE['instance_az']})")

# Security group: no inbound; outbound defaults are kept.
try:
    sg = ec2.create_security_group(
        GroupName=SG_NAME,
        Description="labrun ebs-volume-types-and-iops shared SG",
        VpcId=default_vpc_id,
        TagSpecifications=[
            {
                "ResourceType": "security-group",
                "Tags": [
                    {"Key": "Name", "Value": SG_NAME},
                    {"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE},
                ],
            }
        ],
    )
    LAB_STATE["sg_id"] = sg["GroupId"]
except ClientError as e:
    if e.response.get("Error", {}).get("Code") == "InvalidGroup.Duplicate":
        sgs = ec2.describe_security_groups(
            Filters=[{"Name": "group-name", "Values": [SG_NAME]}],
        )["SecurityGroups"]
        LAB_STATE["sg_id"] = sgs[0]["GroupId"]
    else:
        raise
print(f"Security group: {LAB_STATE['sg_id']}")

# Resolve the latest AL2023 ARM AMI via SSM.
ami_id = ssm.get_parameter(
    Name="/aws/service/ami-amazon-linux-latest/al2023-ami-kernel-default-arm64",
)["Parameter"]["Value"]
print(f"AL2023 ARM AMI: {ami_id}")

# IAM eventual consistency: wait ~15 seconds before run_instances so the
# instance profile is visible to EC2.
print("Waiting 15s for IAM instance profile to propagate. Your IAM role is stuck in traffic.")
time.sleep(15)

# YOUR CODE: call ec2.create_volume to provision the 8 GiB io2 volume in
# LAB_STATE["instance_az"] at exactly 1000 IOPS. Required arguments:
#   VolumeType="io2", Size=8, Iops=1000,
#   AvailabilityZone=LAB_STATE["instance_az"],
#   TagSpecifications=[{
#       "ResourceType": "volume",
#       "Tags": [
#           {"Key": "Name", "Value": IO2_NAME},
#           {"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE},
#       ],
#   }]
# Capture the returned VolumeId in LAB_STATE["io2_volume_id"].

# YOUR CODE: call ec2.run_instances to launch the t4g.small with the gp3
# inline via BlockDeviceMappings. Required arguments:
#   ImageId=ami_id, InstanceType="t4g.small", MinCount=1, MaxCount=1,
#   SubnetId=subnet_id,
#   SecurityGroupIds=[LAB_STATE["sg_id"]],
#   IamInstanceProfile={"Name": INSTANCE_PROFILE_NAME},
#   BlockDeviceMappings=[{
#       "DeviceName": GP3_DEVICE,
#       "Ebs": {
#           "VolumeType": "gp3", "VolumeSize": 8,
#           "DeleteOnTermination": True,
#       },
#   }],
#   TagSpecifications=[
#       {"ResourceType": "instance", "Tags": [
#           {"Key": "Name", "Value": EC2_NAME},
#           {"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE},
#       ]},
#       {"ResourceType": "volume", "Tags": [
#           {"Key": "Name", "Value": GP3_NAME},
#           {"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE},
#       ]},
#   ],
# Capture the new InstanceId in LAB_STATE["instance_id"].

_rebuild_manifest()
print(f"io2 volume: {LAB_STATE['io2_volume_id']}")
print(f"Instance:   {LAB_STATE['instance_id']}")
print()
print("Run the observe cell below to wait for the instance to reach running,")
print("then this cell's next phase attaches the io2 at /dev/sdg.")

In [ ]:
# NBVAL_SKIP
# Observe: poll until the instance state is running, then attach io2 at
# /dev/sdg. Then poll until status check is ok and both volumes show
# State=in-use. Ceiling 6 minutes.

ec2_obs = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

io2_attached = False
deadline = time.time() + 360
while time.time() < deadline:
    clear_output(wait=True)
    elapsed = int(360 - (deadline - time.time()))
    try:
        r = ec2_obs.describe_instances(InstanceIds=[LAB_STATE["instance_id"]])
        inst = r["Reservations"][0]["Instances"][0]
        state = inst["State"]["Name"]
    except Exception as e:
        state = f"err: {e}"
    try:
        ss = ec2_obs.describe_instance_status(
            InstanceIds=[LAB_STATE["instance_id"]], IncludeAllInstances=True
        )["InstanceStatuses"][0]
        status = ss["InstanceStatus"]["Status"]
        system = ss["SystemStatus"]["Status"]
    except Exception:
        status = "n/a"
        system = "n/a"

    try:
        gp3v = ec2_obs.describe_volumes(
            Filters=[
                {"Name": "attachment.instance-id", "Values": [LAB_STATE["instance_id"]]},
                {"Name": "attachment.device", "Values": [GP3_DEVICE]},
            ],
        )["Volumes"]
        if gp3v:
            LAB_STATE["gp3_volume_id"] = gp3v[0]["VolumeId"]
            gp3_state = gp3v[0].get("State")
        else:
            gp3_state = "missing"
    except ClientError:
        gp3_state = "err"

    try:
        io2v = ec2_obs.describe_volumes(VolumeIds=[LAB_STATE["io2_volume_id"]])["Volumes"][0]
        io2_state = io2v.get("State")
    except ClientError:
        io2_state = "missing"

    print(f"Instance + volumes status at t+{elapsed:>3}s:")
    print("-" * 70)
    print(f"  Instance:    {LAB_STATE['instance_id']}  state={state}")
    print(f"  Status:      InstanceStatus={status}  SystemStatus={system}")
    print(f"  gp3 volume:  {LAB_STATE.get('gp3_volume_id') or '?'}  state={gp3_state}")
    print(f"  io2 volume:  {LAB_STATE['io2_volume_id']}  state={io2_state}")

    if state == "running" and not io2_attached:
        try:
            ec2_obs.attach_volume(
                VolumeId=LAB_STATE["io2_volume_id"],
                InstanceId=LAB_STATE["instance_id"],
                Device=IO2_DEVICE,
            )
            io2_attached = True
            print()
            print(f"io2 attached at {IO2_DEVICE}; waiting for in-use state.")
        except ClientError as e:
            if e.response.get("Error", {}).get("Code") == "VolumeInUse":
                io2_attached = True
            else:
                print(f"attach_volume error: {e}")

    if (
        state == "running"
        and status == "ok"
        and gp3_state == "in-use"
        and io2_state == "in-use"
    ):
        print()
        print("Instance is running with both volumes attached and status checks ok.")
        break
    time.sleep(15)
else:
    print()
    print("Timed out waiting for the instance + volumes to settle. Check the EC2 console.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: EC2 instance is running with both gp3 and io2 volumes
# attached at the expected device names.

def checkpoint_1(session):
    try:
        ec2c = boto3.client(
            "ec2",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        insts = ec2c.describe_instances(
            Filters=[
                {"Name": f"tag:{LAB_TAG_KEY}", "Values": [LAB_TAG_VALUE]},
                {"Name": "instance-state-name", "Values": ["pending", "running"]},
            ],
        ).get("Reservations", [])
        flat = [i for r in insts for i in r.get("Instances", [])]
        if len(flat) != 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Expected exactly 1 running instance tagged {LAB_TAG_KEY}={LAB_TAG_VALUE}, "
                    f"found {len(flat)}."
                ),
            )
        inst = flat[0]
        if inst["State"]["Name"] != "running":
            return CheckpointResult(
                status="fail",
                error_reason=f"Instance is in state {inst['State']['Name']!r}, expected running.",
            )

        block_devices = {bdm["DeviceName"]: bdm for bdm in inst.get("BlockDeviceMappings", [])}
        if GP3_DEVICE not in block_devices:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Instance has no block device at {GP3_DEVICE} (the gp3 mount). "
                    f"Found devices: {sorted(block_devices.keys())}."
                ),
            )
        if IO2_DEVICE not in block_devices:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Instance has no block device at {IO2_DEVICE} (the io2 mount). "
                    f"Run attach_volume(VolumeId=io2_volume_id, Device='{IO2_DEVICE}')."
                ),
            )

        gp3_id = block_devices[GP3_DEVICE]["Ebs"]["VolumeId"]
        io2_id = block_devices[IO2_DEVICE]["Ebs"]["VolumeId"]
        LAB_STATE["gp3_volume_id"] = gp3_id
        LAB_STATE["io2_volume_id"] = io2_id

        vols = ec2c.describe_volumes(VolumeIds=[gp3_id, io2_id])["Volumes"]
        for v in vols:
            if v.get("State") != "in-use":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Volume {v.get('VolumeId')} state is {v.get('State')!r}, "
                        f"expected in-use."
                    ),
                )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Two boto3 calls live behind the YOUR CODE blocks. The first is `create_volume` for the io2; the second is `run_instances` with the gp3 inline via `BlockDeviceMappings`. The lab pre-populated every argument you need in the comments above each block; you are wiring them through.

</details>

<details><summary>Hint 2 (direction)</summary>

`create_volume` returns a dict with a top-level `VolumeId` field. `run_instances` returns `{"Instances": [...]}`; the new ID lives at `["Instances"][0]["InstanceId"]`. Pass `IamInstanceProfile={"Name": INSTANCE_PROFILE_NAME}` (not Arn) so EC2 looks up the profile by name. The io2's `AvailabilityZone` must match `LAB_STATE["instance_az"]` or attach will fail later with `InvalidParameterValue` across AZs.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
io2 = ec2.create_volume(
    VolumeType="io2",
    Size=8,
    Iops=1000,
    AvailabilityZone=LAB_STATE["instance_az"],
    TagSpecifications=[{
        "ResourceType": "volume",
        "Tags": [
            {"Key": "Name", "Value": IO2_NAME},
            {"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE},
        ],
    }],
)
LAB_STATE["io2_volume_id"] = io2["VolumeId"]

resp = ec2.run_instances(
    ImageId=ami_id,
    InstanceType="t4g.small",
    MinCount=1,
    MaxCount=1,
    SubnetId=subnet_id,
    SecurityGroupIds=[LAB_STATE["sg_id"]],
    IamInstanceProfile={"Name": INSTANCE_PROFILE_NAME},
    BlockDeviceMappings=[{
        "DeviceName": GP3_DEVICE,
        "Ebs": {
            "VolumeType": "gp3",
            "VolumeSize": 8,
            "DeleteOnTermination": True,
        },
    }],
    TagSpecifications=[
        {"ResourceType": "instance", "Tags": [
            {"Key": "Name", "Value": EC2_NAME},
            {"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE},
        ]},
        {"ResourceType": "volume", "Tags": [
            {"Key": "Name", "Value": GP3_NAME},
            {"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE},
        ]},
    ],
)
LAB_STATE["instance_id"] = resp["Instances"][0]["InstanceId"]
```

</details>

**Wallet check.** Instance just started. t4g.small bills at $0.0168/hour, or about 0.028 cents per minute. The two volumes prorate to about $0.0023 combined for a one-hour session. Running total: half a cent so far.

## Task 2: Verify both volumes carry the spec you asked for

Before you benchmark, prove the volumes match the spec. The published gp3 default at 8 GiB is 3000 IOPS baseline and 125 MB/s throughput, and AWS returns those values on `describe_volumes` even when you did not pass `Iops=` explicitly at create time. The io2 returns exactly the `Iops=1000` you provisioned.

This task is a single `describe_volumes` call followed by a field walk. The checkpoint is the structured assertion.

In [ ]:
# NBVAL_SKIP
# Task 2: read both volume specs and print them side by side.

ec2 = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: call ec2.describe_volumes(VolumeIds=[
#   LAB_STATE["gp3_volume_id"], LAB_STATE["io2_volume_id"]]) and pull the
# two Volume dicts into gp3v and io2v.

# Replace these placeholder dicts with the real describe_volumes data.
gp3v = {"VolumeType": "?", "Size": 0, "Iops": 0, "Throughput": 0}
io2v = {"VolumeType": "?", "Size": 0, "Iops": 0}

print("gp3 volume specs:")
print(f"  Id:         {LAB_STATE['gp3_volume_id']}")
print(f"  VolumeType: {gp3v.get('VolumeType')}")
print(f"  Size:       {gp3v.get('Size')} GiB")
print(f"  Iops:       {gp3v.get('Iops')}")
print(f"  Throughput: {gp3v.get('Throughput')} MB/s")
print()
print("io2 volume specs:")
print(f"  Id:         {LAB_STATE['io2_volume_id']}")
print(f"  VolumeType: {io2v.get('VolumeType')}")
print(f"  Size:       {io2v.get('Size')} GiB")
print(f"  Iops:       {io2v.get('Iops')}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: gp3 has 3000 IOPS / 125 MB/s defaults at 8 GiB; io2 has
# exactly 1000 provisioned IOPS at 8 GiB.

def checkpoint_2(session):
    try:
        ec2c = boto3.client(
            "ec2",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        if not LAB_STATE.get("gp3_volume_id") or not LAB_STATE.get("io2_volume_id"):
            return CheckpointResult(
                status="fail",
                error_reason="Volume IDs not captured. Re-run Task 1 + observe + Checkpoint 1.",
            )
        vols = ec2c.describe_volumes(
            VolumeIds=[LAB_STATE["gp3_volume_id"], LAB_STATE["io2_volume_id"]]
        )["Volumes"]
        by_type = {v["VolumeType"]: v for v in vols}

        if "gp3" not in by_type:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "gp3 volume not found among the two volumes attached. "
                    "BlockDeviceMappings.Ebs.VolumeType must be 'gp3'."
                ),
            )
        gp3 = by_type["gp3"]
        if gp3.get("Size") != 8:
            return CheckpointResult(
                status="fail",
                error_reason=f"gp3 size is {gp3.get('Size')} GiB, expected 8 GiB.",
            )
        if gp3.get("Iops") != 3000:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"gp3 Iops is {gp3.get('Iops')}, expected the gp3 default 3000 at 8 GiB. "
                    f"Did you set a custom Iops on the BlockDeviceMappings Ebs entry?"
                ),
            )
        if gp3.get("Throughput") != 125:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"gp3 Throughput is {gp3.get('Throughput')} MB/s, expected the gp3 default 125 at 8 GiB."
                ),
            )

        if "io2" not in by_type:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "io2 volume not found. VolumeType in create_volume must be 'io2', "
                    "not 'gp3' or 'io1'."
                ),
            )
        io2 = by_type["io2"]
        if io2.get("Size") != 8:
            return CheckpointResult(
                status="fail",
                error_reason=f"io2 size is {io2.get('Size')} GiB, expected 8 GiB.",
            )
        if io2.get("Iops") != 1000:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"io2 Iops is {io2.get('Iops')}, expected exactly 1000. "
                    f"create_volume must pass Iops=1000."
                ),
            )

        tags_ok = True
        for v in (gp3, io2):
            tagset = {t["Key"]: t["Value"] for t in v.get("Tags", [])}
            if tagset.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
                tags_ok = False
        if not tags_ok:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"At least one volume is missing {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Tag both volumes at create time so the orphan scan finds them."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

One boto3 call: `describe_volumes(VolumeIds=[...])`. It returns a `Volumes` list. The two volumes have different `VolumeType` fields, which is the easiest way to tell them apart in code.

</details>

<details><summary>Hint 2 (direction)</summary>

`describe_volumes(VolumeIds=[gp3_id, io2_id])["Volumes"]` returns the two volumes in some order (not guaranteed). Build a `{VolumeType: volume_dict}` map so you can index by type rather than by list position. Pull `gp3v = by_type["gp3"]` and `io2v = by_type["io2"]`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
resp = ec2.describe_volumes(
    VolumeIds=[LAB_STATE["gp3_volume_id"], LAB_STATE["io2_volume_id"]]
)
by_type = {v["VolumeType"]: v for v in resp["Volumes"]}
gp3v = by_type["gp3"]
io2v = by_type["io2"]
```

</details>

**Wallet check.** Still about half a cent in. `describe_volumes` is free; no metered API call here.

## Task 3: Run fio against the gp3 volume via SSM Session Manager

The benchmark is what settles the debate. You drive `fio` against the raw gp3 device (`/dev/nvme1n1`, the Nitro remap of `/dev/sdf`) with a 4 KiB random-read workload, 32 outstanding I/Os, and a 60-second runtime. fio emits JSON; you parse the sustained IOPS off `jobs[0].read.iops`.

SSM Session Manager is the in-instance shell channel. The `AmazonSSMManagedInstanceCore` policy on the instance role plus the SSM agent (pre-installed on AL2023) is everything you need. `ssm.send_command` returns a CommandId; the observe cell polls `get_command_invocation` until the command finishes and prints the output.

AL2023 ships with `fio` available via `dnf install` and the `--readonly` flag means fio reads from the raw device without writing or trashing data.

In [ ]:
# NBVAL_SKIP
# Task 3: build the SSM command and call send_command to benchmark gp3.

ssm = boto3.client(
    "ssm",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

GP3_FIO_COMMAND = (
    "sudo dnf install -y fio >/dev/null && "
    f"sudo fio --name=randread --filename={GP3_NITRO_DEVICE} "
    "--readonly --rw=randread --bs=4k --iodepth=32 --runtime=60 "
    "--time_based --group_reporting --output-format=json"
)

# YOUR CODE: call ssm.send_command(
#     InstanceIds=[LAB_STATE["instance_id"]],
#     DocumentName="AWS-RunShellScript",
#     Parameters={"commands": [GP3_FIO_COMMAND]},
#     TimeoutSeconds=300,
# ) and capture the CommandId in gp3_command_id. The observe cell below
# polls until the command finishes and prints the parsed IOPS.

gp3_command_id = None  # YOUR CODE replaces this with the send_command result.

print(f"SSM command queued: {gp3_command_id}")
print("Run the observe cell below to wait for fio to finish (~75 seconds).")

In [ ]:
# NBVAL_SKIP
# Observe: poll get_command_invocation until the gp3 fio run finishes,
# parse the JSON, and stash the IOPS in LAB_STATE.

ssm_obs = boto3.client(
    "ssm",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deadline = time.time() + 240
fio_status = "Pending"
while time.time() < deadline:
    clear_output(wait=True)
    elapsed = int(240 - (deadline - time.time()))
    try:
        inv = ssm_obs.get_command_invocation(
            CommandId=gp3_command_id,
            InstanceId=LAB_STATE["instance_id"],
        )
        fio_status = inv.get("Status")
        stdout = inv.get("StandardOutputContent", "")
        stderr = inv.get("StandardErrorContent", "")
    except ClientError as e:
        fio_status = f"err: {e}"
        stdout = ""
        stderr = ""

    print(f"gp3 fio benchmark at t+{elapsed:>3}s (asking Athena to put on her reading glasses):")
    print("-" * 70)
    print(f"  Command:    {gp3_command_id}")
    print(f"  Status:     {fio_status}")
    print(f"  Instance:   {LAB_STATE['instance_id']}")

    if fio_status in ("Success",):
        try:
            parsed = json.loads(stdout)
            iops = parsed["jobs"][0]["read"]["iops"]
            LAB_STATE["gp3_fio_iops"] = iops
            print(f"  Sustained IOPS: {iops:.1f} (4 KiB random read, iodepth=32, 60s)")
            print()
            print("gp3 benchmark complete. Move on to the checkpoint cell.")
        except (ValueError, KeyError, IndexError) as e:
            print(f"  Could not parse fio JSON: {e}")
            print("  Raw stdout (first 800 chars):")
            print(stdout[:800])
        break
    if fio_status in ("Cancelled", "TimedOut", "Failed"):
        print()
        print(f"SSM command entered terminal {fio_status!r} state.")
        if stderr:
            print("stderr (first 800 chars):")
            print(stderr[:800])
        break
    time.sleep(10)
else:
    print()
    print("Timed out before fio finished. Re-run the observe cell.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: gp3 fio sustained IOPS lands in [2500, 3500].

def checkpoint_3(session):
    try:
        iops = LAB_STATE.get("gp3_fio_iops")
        if iops is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "gp3 IOPS not captured yet. Run the observe cell above until "
                    "the fio JSON is parsed and LAB_STATE['gp3_fio_iops'] is set."
                ),
            )
        if iops < 2500:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"gp3 sustained {iops:.1f} IOPS, below the 2500 floor. gp3 at 8 GiB "
                    f"should deliver close to its 3000 IOPS baseline; if you fell short "
                    f"the iodepth or runtime may be too low for the curve to plateau."
                ),
            )
        if iops > 3500:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"gp3 sustained {iops:.1f} IOPS, above the 3500 ceiling. Did the "
                    f"BlockDeviceMappings request a custom Iops above 3000? gp3 at 8 GiB "
                    f"should sit at the default 3000 baseline."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

`ssm.send_command` is one boto3 call. It needs the instance ID, the document name `AWS-RunShellScript`, the `commands` parameter as a list of one string, and a timeout. The fio command string is already assembled in `GP3_FIO_COMMAND`; you do not write the shell.

</details>

<details><summary>Hint 2 (direction)</summary>

`ssm.send_command(...)` returns `{"Command": {"CommandId": "...", ...}}`. Pull the ID off `["Command"]["CommandId"]` and stash it in `gp3_command_id`. The observe cell hits `get_command_invocation` against that ID and the same instance.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
resp = ssm.send_command(
    InstanceIds=[LAB_STATE["instance_id"]],
    DocumentName="AWS-RunShellScript",
    Parameters={"commands": [GP3_FIO_COMMAND]},
    TimeoutSeconds=300,
)
gp3_command_id = resp["Command"]["CommandId"]
```

</details>

**Wallet check.** SSM `send_command` and `get_command_invocation` are free. fio inside the instance is just CPU and disk; it does not add a separate bill. Running total: less than a cent.

## Task 4: Run fio against the io2 volume the same way

Same workload, different volume. fio runs against `/dev/nvme2n1` (the Nitro remap of `/dev/sdg`) with identical parameters: 4 KiB random read, 32 outstanding I/Os, 60-second runtime. The expected result lands near 1000 IOPS, matching the provisioned spec.

This is the part that ends the debate: io2 delivered exactly what you paid for. Not 1500. Not 3000. 1000. Provisioned IOPS is a contract.

In [ ]:
# NBVAL_SKIP
# Task 4: send the io2 fio benchmark via SSM.

ssm = boto3.client(
    "ssm",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

IO2_FIO_COMMAND = (
    f"sudo fio --name=randread --filename={IO2_NITRO_DEVICE} "
    "--readonly --rw=randread --bs=4k --iodepth=32 --runtime=60 "
    "--time_based --group_reporting --output-format=json"
)

# YOUR CODE: call ssm.send_command with InstanceIds=[LAB_STATE["instance_id"]],
# DocumentName="AWS-RunShellScript", Parameters={"commands": [IO2_FIO_COMMAND]},
# TimeoutSeconds=300. Capture the CommandId in io2_command_id.

io2_command_id = None  # YOUR CODE replaces this with the send_command result.

print(f"SSM command queued: {io2_command_id}")
print("Run the observe cell below to wait for fio to finish (~75 seconds).")

In [ ]:
# NBVAL_SKIP
# Observe: poll get_command_invocation until the io2 fio run finishes,
# parse the JSON, stash the IOPS.

ssm_obs = boto3.client(
    "ssm",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

deadline = time.time() + 180
fio_status = "Pending"
while time.time() < deadline:
    clear_output(wait=True)
    elapsed = int(180 - (deadline - time.time()))
    try:
        inv = ssm_obs.get_command_invocation(
            CommandId=io2_command_id,
            InstanceId=LAB_STATE["instance_id"],
        )
        fio_status = inv.get("Status")
        stdout = inv.get("StandardOutputContent", "")
        stderr = inv.get("StandardErrorContent", "")
    except ClientError as e:
        fio_status = f"err: {e}"
        stdout = ""
        stderr = ""

    print(f"io2 fio benchmark at t+{elapsed:>3}s (poking your provisioned IOPS to see if it woke up):")
    print("-" * 70)
    print(f"  Command:    {io2_command_id}")
    print(f"  Status:     {fio_status}")
    print(f"  Instance:   {LAB_STATE['instance_id']}")

    if fio_status in ("Success",):
        try:
            parsed = json.loads(stdout)
            iops = parsed["jobs"][0]["read"]["iops"]
            LAB_STATE["io2_fio_iops"] = iops
            print(f"  Sustained IOPS: {iops:.1f} (4 KiB random read, iodepth=32, 60s)")
            print()
            print("Side-by-side measurement:")
            print(f"  gp3 sustained: {LAB_STATE.get('gp3_fio_iops', 'n/a'):.1f} IOPS (baseline 3000)")
            print(f"  io2 sustained: {iops:.1f} IOPS (provisioned 1000)")
        except (ValueError, KeyError, IndexError) as e:
            print(f"  Could not parse fio JSON: {e}")
            print("  Raw stdout (first 800 chars):")
            print(stdout[:800])
        break
    if fio_status in ("Cancelled", "TimedOut", "Failed"):
        print()
        print(f"SSM command entered terminal {fio_status!r} state.")
        if stderr:
            print("stderr (first 800 chars):")
            print(stderr[:800])
        break
    time.sleep(10)
else:
    print()
    print("Timed out before fio finished. Re-run the observe cell.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: io2 fio sustained IOPS lands in [800, 1200].

def checkpoint_4(session):
    try:
        iops = LAB_STATE.get("io2_fio_iops")
        if iops is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "io2 IOPS not captured yet. Run the observe cell above until "
                    "the fio JSON is parsed and LAB_STATE['io2_fio_iops'] is set."
                ),
            )
        if iops < 800:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"io2 sustained {iops:.1f} IOPS, below the 800 floor. io2 at 1000 "
                    f"provisioned IOPS should deliver close to 1000; if you came in low "
                    f"the iodepth or workload mix may not be saturating the volume."
                ),
            )
        if iops > 1200:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"io2 sustained {iops:.1f} IOPS, above the 1200 ceiling. Did create_volume "
                    f"request more than 1000 provisioned IOPS? io2 returns exactly the "
                    f"Iops value you provisioned."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Same shape as Task 3's `send_command`. The only thing that changes is the `commands` list: it now points at `IO2_FIO_COMMAND` and the `/dev/nvme2n1` device.

</details>

<details><summary>Hint 2 (direction)</summary>

Reuse the exact pattern from Task 3. `ssm.send_command(...)` returns `{"Command": {"CommandId": "..."}}`. Capture `["Command"]["CommandId"]` into `io2_command_id`, then run the observe cell to wait for completion.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
resp = ssm.send_command(
    InstanceIds=[LAB_STATE["instance_id"]],
    DocumentName="AWS-RunShellScript",
    Parameters={"commands": [IO2_FIO_COMMAND]},
    TimeoutSeconds=300,
)
io2_command_id = resp["Command"]["CommandId"]
```

</details>

**Wallet check.** Still about 1.5 cents. Two SSM commands, two fio runs, zero new line items. The next cell is the one that matters: cleanup.

## Cleanup

This is the one that matters. The instance is the hourly meter; the io2 volume is the trap that bills indefinitely if the instance terminates without detaching it. The cell below detaches the io2 first, deletes it, then terminates the instance (which auto-deletes the gp3 via DeleteOnTermination=True). The instance profile and role come down last. Re-running this cell is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. The io2 volume has no labrun-checks adapter in 0.6.0, so the
# cell detaches and deletes it manually before run_cleanup terminates
# the instance. The gp3 has DeleteOnTermination=True so it vanishes with
# the instance.

import sys

_detach_and_delete_io2()
print("io2 detached and delete requested.")

_rebuild_manifest()
result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if r.type == "ec2_instance")
critical_destroyed = sum(
    1 for r in CLEANUP_MANIFEST
    if r.type == "ec2_instance" and r.id not in failed_ids
)
standard_total = len(CLEANUP_MANIFEST) - critical_total
standard_destroyed = standard_total - sum(
    1 for rid in failed_ids for r in CLEANUP_MANIFEST
    if r.id == rid and r.type != "ec2_instance"
)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.02 to $0.05.** t4g.small at $0.0168/hour, gp3 plus io2 storage and provisioned IOPS prorated for the session, and a single Cost Explorer call's worth of zero (this lab does not touch CE). The cleanup cell stopped the meter the moment it printed `Cleanup complete`.

## Reflection

These are not graded. They are for you.

1. Walk through the EBS volume types (gp3, gp2, io1, io2, io2 Block Express, st1, sc1) and group them by what they optimize for: cost per GB, sustained IOPS, throughput, latency. Which workload pattern fits each, and where do gp3 and io2 sit on the IOPS-versus-cost frontier?

2. Your team is debating whether to use a single io2 Block Express volume at 256000 IOPS or four gp3 volumes in a RAID0 stripe each at 16000 IOPS (64000 IOPS aggregate). Walk through the trade-offs in terms of cost, durability, blast radius if one volume fails, and operational complexity. When is each the right call?

## Exam-Style Practice

**Question 1 (Domain 3, EBS volume-type selection by constraint):**

You are designing storage for a write-heavy OLTP database that requires:

- Sustained 25,000 IOPS at sub-millisecond latency.
- 99.999% volume durability.
- A single block device, not striped RAID.

Which EBS volume type fits all three requirements?

A. gp3 provisioned at 25,000 IOPS.

B. io2 provisioned at 25,000 IOPS.

C. io2 Block Express provisioned at 25,000 IOPS.

D. st1 with 1 TB size to maximize throughput.

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: gp3 maxes out at 16,000 IOPS per volume. To reach 25,000 IOPS on gp3 you would need RAID0 across multiple volumes, which the question explicitly forbids.
- B is wrong on latency: standard io2 delivers single-digit-millisecond latency, not sub-millisecond. It is durable enough (99.999%) and supports the IOPS, but the sub-millisecond requirement is the constraint that disqualifies it.
- C is correct: io2 Block Express is the AWS-recommended choice for sub-millisecond latency at high IOPS. It supports up to 256,000 IOPS, 99.999% durability, and sub-millisecond latency on Nitro instances. It is the only single-volume option that meets all three constraints.
- D is wrong: st1 is a throughput-optimized HDD designed for big sequential workloads. It does not provide high IOPS and is not designed for OLTP latency.

</details>

**Question 2 (Domain 3, gp3 vs io2 cost-per-IOPS trade-off):**

A team needs 3000 sustained IOPS on a 100 GiB volume. They are choosing between gp3 (priced at $0.08/GB-month, IOPS up to 3000 free at this size, additional IOPS at $0.005 each) and io2 (priced at $0.125/GB-month plus $0.065 per provisioned IOPS-month). Which volume type is cheaper monthly, and by approximately how much?

A. gp3 by ~$200/month (gp3 storage $8, io2 storage $12.50 + 3000 IOPS at $0.065 = $195+).

B. io2 by ~$5/month (io2's better durability justifies a price premium).

C. They cost the same because both deliver 3000 IOPS at this size.

D. gp3 by ~$5/month (gp3 is slightly cheaper but the gap is operational, not financial).

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: gp3 at 100 GiB delivers 3000 IOPS baseline at the storage price alone ($8/month). io2 at 100 GiB charges $12.50/month for storage PLUS $195/month for 3000 provisioned IOPS (3000 x $0.065). io2 is approximately $207/month; gp3 is $8/month. The gp3-to-io2 cost-per-IOPS premium is the single biggest reason teams choose gp3 by default and reach for io2 only when they cannot get the workload's IOPS or latency profile from gp3.
- B is wrong: io2 is dramatically more expensive at this size and IOPS profile, not $5 cheaper. io2's durability advantage (99.999% vs gp3's 99.8-99.9%) is real but is not 'the same cost' or 'io2 is cheaper.'
- C is wrong: the volumes have very different pricing models and io2's per-IOPS charge dominates at 3000 IOPS.
- D is wrong: gp3 is cheaper, but by ~$200/month, not $5. The magnitude matters here because the gp3-to-io2 gap is the entire reason teams default to gp3.

</details>